In [3]:
import torch

pt_path = r"models/8300/8300_best_bilstm.pt"

print(f"\n{'='*60}")
print(f"Inspecting: {pt_path}")
print(f"{'='*60}")

checkpoint = torch.load(pt_path, map_location="cpu")

print(f"\n[Type] {type(checkpoint)}")

if isinstance(checkpoint, dict):
    print(f"\n[Keys] {list(checkpoint.keys())}")
    for k, v in checkpoint.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k}: Tensor {tuple(v.shape)} dtype={v.dtype}")
        elif isinstance(v, dict):
            print(f"  {k}: dict with {len(v)} keys")
        else:
            print(f"  {k}: {type(v).__name__} = {v}")

    state_dict = None
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
        state_dict = checkpoint

    if state_dict:
        print(f"\n[State Dict Layers]")
        for name, tensor in state_dict.items():
            print(f"  {name:50s} {str(tuple(tensor.shape)):30s} {tensor.dtype}")

elif hasattr(checkpoint, "state_dict"):
    print("\n[Full model object detected]")
    print(f"  Class: {type(checkpoint).__name__}")
    print(f"\n[State Dict Layers]")
    for name, tensor in checkpoint.state_dict().items():
        print(f"  {name:50s} {str(tuple(tensor.shape)):30s} {tensor.dtype}")

else:
    print(f"\n[Raw value] {checkpoint}")

print(f"\n{'='*60}\n")


Inspecting: models/8300/8300_best_bilstm.pt

[Type] <class 'collections.OrderedDict'>

[Keys] ['pos_bias', 'output_blend', 'lstm.weight_ih_l0', 'lstm.weight_hh_l0', 'lstm.bias_ih_l0', 'lstm.bias_hh_l0', 'lstm.weight_ih_l0_reverse', 'lstm.weight_hh_l0_reverse', 'lstm.bias_ih_l0_reverse', 'lstm.bias_hh_l0_reverse', 'lstm.weight_ih_l1', 'lstm.weight_hh_l1', 'lstm.bias_ih_l1', 'lstm.bias_hh_l1', 'lstm.weight_ih_l1_reverse', 'lstm.weight_hh_l1_reverse', 'lstm.bias_ih_l1_reverse', 'lstm.bias_hh_l1_reverse', 'attention.weight', 'attention.bias', 'ln.weight', 'ln.bias', 'fc.weight', 'fc.bias', 'skip_fc.weight', 'skip_fc.bias']
  pos_bias: Tensor (1,) dtype=torch.float32
  output_blend: Tensor () dtype=torch.float32
  lstm.weight_ih_l0: Tensor (512, 42) dtype=torch.float32
  lstm.weight_hh_l0: Tensor (512, 128) dtype=torch.float32
  lstm.bias_ih_l0: Tensor (512,) dtype=torch.float32
  lstm.bias_hh_l0: Tensor (512,) dtype=torch.float32
  lstm.weight_ih_l0_reverse: Tensor (512, 42) dtype=torch.f

C:\Users\User\AppData\Local\Temp\ipykernel_25800\643577547.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pt_path, map_location="cpu")


In [4]:
import torch
import torch.nn as nn
import onnx
import os

symbols = ["8300", "1010", "8100", "2050", "2060", "1180", "3080", "3090"]

# ── Model Definition ──────────────────────────────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.attention    = nn.Linear(hidden_size * 2, 1)
        self.pos_bias     = nn.Parameter(torch.zeros(1))
        self.ln           = nn.LayerNorm(hidden_size * 2)
        self.dropout      = nn.Dropout(dropout)
        self.fc           = nn.Linear(hidden_size * 2, 2)
        self.skip_fc      = nn.Linear(hidden_size * 2, 2)
        self.output_blend = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        seq_len      = lstm_out.size(1)
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        out_attn     = self.fc(self.dropout(self.ln(context)))
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


def infer_hyperparams(state_dict: dict) -> dict:
    """
    Infer model hyperparameters directly from state dict tensor shapes.

    Key derivations:
      lstm.weight_ih_l0  : shape (4*hidden, input_size)
        → hidden_size = shape[0] // 4
        → input_size  = shape[1]
      lstm.weight_ih_lN  : presence of layer index N determines num_layers
    """
    ih_l0      = state_dict["lstm.weight_ih_l0"]   # (4*H, input_size)
    hidden_size = ih_l0.shape[0] // 4
    input_size  = ih_l0.shape[1]

    # Count how many weight_ih layers exist  (l0, l1, l2 …)
    num_layers = sum(
        1 for k in state_dict
        if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    return {
        "input_size":  input_size,
        "hidden_size": hidden_size,
        "num_layers":  num_layers,
    }


def convert(symbol: str):
    pt_path  = f"models/{symbol}/{symbol}_best_bilstm.pt"
    onnx_dir = f"models/{symbol}"
    onnx_path = f"{onnx_dir}/{symbol}_bilstm.onnx"

    if not os.path.exists(pt_path):
        print(f"[SKIP] {symbol}: file not found → {pt_path}")
        return

    print(f"\n[{symbol}] Loading {pt_path} ...")

    state_dict = torch.load(pt_path, map_location="cpu", weights_only=True)

    # Unwrap checkpoint dicts if needed
    if "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]
    elif "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    # ── Infer hyperparameters from weight shapes ───────────────────────────
    hp = infer_hyperparams(state_dict)
    print(f"[{symbol}] Inferred → input_size={hp['input_size']}, "
          f"hidden_size={hp['hidden_size']}, num_layers={hp['num_layers']}")

    # ── Build model and load weights ───────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = hp["input_size"],
        hidden_size = hp["hidden_size"],
        num_layers  = hp["num_layers"],
    )
    model.load_state_dict(state_dict)
    model.eval()

    # ── Dummy input: (batch=1, seq=1, input_size) ──────────────────────────
    # seq_len=1 is fine — dynamic axes allow any length at inference time
    dummy_input = torch.randn(1, 1, hp["input_size"])

    os.makedirs(onnx_dir, exist_ok=True)

    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        opset_version=17,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input":  {0: "batch_size", 1: "seq_len"},
            "output": {0: "batch_size"},
        },
        do_constant_folding=True,
    )

    # ── Validate ───────────────────────────────────────────────────────────
    onnx.checker.check_model(onnx.load(onnx_path))
    print(f"[{symbol}] ✓ Saved & validated → {onnx_path}")


# ── Run all conversions ───────────────────────────────────────────────────────
if __name__ == "__main__":
    for sym in symbols:
        try:
            convert(sym)
        except Exception as e:
            print(f"[{sym}] ✗ Error: {e}")


[8300] Loading models/8300/8300_best_bilstm.pt ...
[8300] Inferred → input_size=42, hidden_size=128, num_layers=2


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


[8300] ✓ Saved & validated → models/8300/8300_bilstm.onnx

[1010] Loading models/1010/1010_best_bilstm.pt ...
[1010] Inferred → input_size=42, hidden_size=128, num_layers=3
[1010] ✓ Saved & validated → models/1010/1010_bilstm.onnx

[8100] Loading models/8100/8100_best_bilstm.pt ...
[8100] Inferred → input_size=42, hidden_size=128, num_layers=2
[8100] ✓ Saved & validated → models/8100/8100_bilstm.onnx

[2050] Loading models/2050/2050_best_bilstm.pt ...
[2050] Inferred → input_size=42, hidden_size=128, num_layers=3
[2050] ✓ Saved & validated → models/2050/2050_bilstm.onnx

[2060] Loading models/2060/2060_best_bilstm.pt ...
[2060] Inferred → input_size=42, hidden_size=64, num_layers=2
[2060] ✓ Saved & validated → models/2060/2060_bilstm.onnx

[1180] Loading models/1180/1180_best_bilstm.pt ...
[1180] Inferred → input_size=42, hidden_size=128, num_layers=3
[1180] ✓ Saved & validated → models/1180/1180_bilstm.onnx

[3080] Loading models/3080/3080_best_bilstm.pt ...
[3080] Inferred → input_si